### **1. Data Cleaning: Shwapno Dataset**

**a. Import pandas :**

In [1]:
import pandas as pd

**b. csv file access for shwapno :**

In [84]:
df_shwapno = pd.read_csv("../data/raw/shwapno_data_1.csv")

**c. Preview data:**

In [3]:
df_shwapno.head()

,Category,Title,Previous_price,Current_price,Unit,Stock
0,Fruits,Meher Shagor Kola (Banana ),৳13,৳13,Per Piece,Add to Bag
1,Fruits,Chini Chompa (Banana) Yellow,৳5.75,৳5.75,Per Piece,Add to Bag
2,Fruits,Joldugui Anarosh (Pineapple) 600gm+ Per Pcs,৳62,৳62,Per Piece,Add to Bag
3,Fruits,Malta (Orange),৳299,Per 1kg,❨Min. 0.50kg ❩,Add to Bag
4,Fruits,Green Apple (Apple Golden Delicious),৳395,Per 1kg,❨Min. 0.50kg ❩,Add to Bag


**d. Structure Overview**

In [4]:
df_shwapno.info()

<class 'pandas.DataFrame'>
RangeIndex: 3890 entries, 0 to 3889
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Category        3890 non-null   str  
 1   Title           3886 non-null   str  
 2   Previous_price  3886 non-null   str  
 3   Current_price   3886 non-null   str  
 4   Unit            3886 non-null   str  
 5   Stock           3890 non-null   str  
dtypes: str(6)
memory usage: 182.5 KB


**e. Remove unnecessary column:** Since we are only keeping available products, we will first drop the "Out of Stock" rows and then remove the stock column itself.

In [5]:
df_shwapno = df_shwapno[df_shwapno['Stock'] != 'out of stock']

In [6]:
df_shwapno = df_shwapno.drop(columns=['Stock']).reset_index(drop=True)

**e.Major cleaning :** Due to inconsistent HTML tags, `unit` data was incorrectly extracted into  `current_price` columns. we resolve misalinements by reassigning the `Unit` data ans update `current_price` with the `Previous_price` values for the affected rows.

In [7]:
def fix_price_unit(df):
    per_unit_mask = df['Current_price'].str.contains('Per', na=False)

    df.loc[per_unit_mask, 'Unit'] = df.loc[per_unit_mask, 'Current_price']
    df.loc[per_unit_mask, 'Current_price'] = df.loc[per_unit_mask, 'Previous_price']

    return df

df_shwapno = fix_price_unit(df_shwapno)

df_shwapno.head()

,Category,Title,Previous_price,Current_price,Unit
0,Fruits,Meher Shagor Kola (Banana ),৳13,৳13,Per Piece
1,Fruits,Chini Chompa (Banana) Yellow,৳5.75,৳5.75,Per Piece
2,Fruits,Joldugui Anarosh (Pineapple) 600gm+ Per Pcs,৳62,৳62,Per Piece
3,Fruits,Malta (Orange),৳299,৳299,Per 1kg
4,Fruits,Green Apple (Apple Golden Delicious),৳395,৳395,Per 1kg


**f. Duplicate check :**

**step-1:** Checking duplicate Records in Dataset:

In [8]:
df_shwapno.duplicated().sum()

np.int64(0)

**step-2:** Checking for product duplication across different categories:

In [9]:
subcat_duplicates_shwapno = df_shwapno[df_shwapno.duplicated(subset=['Title', 'Unit', 'Previous_price', 'Current_price'], keep=False)]

print(subcat_duplicates_shwapno['Category'].value_counts())

Series([], Name: count, dtype: int64)


In [10]:
df_shwapno.info()

<class 'pandas.DataFrame'>
RangeIndex: 3874 entries, 0 to 3873
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Category        3874 non-null   str  
 1   Title           3874 non-null   str  
 2   Previous_price  3874 non-null   str  
 3   Current_price   3874 non-null   str  
 4   Unit            3874 non-null   str  
dtypes: str(5)
memory usage: 151.5 KB


**f. Null checking:**

In [11]:
df_shwapno.isnull().sum()

Category          0
Title             0
Previous_price    0
Current_price     0
Unit              0
dtype: int64

**g. Currency Normalization & Type Conversion:** 

For calculation purpose, Add a new `currency column` and convert `previous_price` & `current_price` from string to float to standardize the unit of measurement:

**i. Add & fix position of `currency column`:**

In [12]:
df_shwapno['Currency'] = df_shwapno['Previous_price'].str[0]

In [13]:
currency_col = df_shwapno.pop('Currency')
df_shwapno.insert(df_shwapno.columns.get_loc('Previous_price'), 'Currency', currency_col)

**ii. Convert `previous_price` and `current_price` from string to float:**

In [14]:
df_shwapno['Previous_price'] = pd.to_numeric(
    df_shwapno['Previous_price'].astype(str)  
                                     .str.replace('৳','', regex=False)
                                     .str.replace(',',''),
    errors='coerce'
)

In [15]:
df_shwapno['Current_price'] = pd.to_numeric(
   df_shwapno['Current_price'].astype(str)
                              .str.replace('৳','',regex=False)
                              .str.replace(',',''),errors='coerce'

)

**h. Final checking:**

In [16]:
df_shwapno.info()

<class 'pandas.DataFrame'>
RangeIndex: 3874 entries, 0 to 3873
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Category        3874 non-null   str    
 1   Title           3874 non-null   str    
 2   Currency        3874 non-null   str    
 3   Previous_price  3874 non-null   float64
 4   Current_price   3874 non-null   float64
 5   Unit            3874 non-null   str    
dtypes: float64(2), str(4)
memory usage: 181.7 KB


**2. Data Cleaning: Chaldal Dataset**

**a. csv file access for Chaldal :**

In [85]:
df_chaldal = pd.read_csv("../data/raw/chaldal_data_2.csv")

**b. Preview data:**

In [18]:
df_chaldal.head()

,Category,Title,Currency,Previous_price,Current_price,Unit
0,Fruits,Shagor Kola (Banana Sagor),৳,55,49,4 pcs
1,Fruits,Banana Chompa (Ready To Eat),৳,35,29,4 pcs
2,Fruits,Bangla Kola,৳,55,45,4 pcs
3,Fruits,Guava Premium (± 50 gm),৳,119,119,1 kg
4,Fruits,Malta ± 50 gm,৳,340,329,1 kg


**c. checking data stucture :**

In [19]:
df_chaldal.info()

<class 'pandas.DataFrame'>
RangeIndex: 4198 entries, 0 to 4197
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Category        4198 non-null   str  
 1   Title           4198 non-null   str  
 2   Currency        4198 non-null   str  
 3   Previous_price  4198 non-null   int64
 4   Current_price   4198 non-null   int64
 5   Unit            4149 non-null   str  
dtypes: int64(2), str(4)
memory usage: 196.9 KB


**d. Duplicate check & drop:**

**step-1:** Check and duplicate Records in Dataset:

In [20]:
df_chaldal.duplicated().sum()

np.int64(482)

In [21]:
df_chaldal = df_chaldal.drop_duplicates().reset_index(drop=True)

**step-2:** Checking for product duplication across different categories:

In [22]:
subcat_duplicates_chaldal = df_chaldal[df_chaldal.duplicated(subset=['Title', 'Unit', 'Previous_price', 'Current_price'], keep=False)]

print(f"Total rows involved in duplicates: {len(subcat_duplicates_chaldal)}")
print(subcat_duplicates_chaldal['Category'].value_counts())

Total rows involved in duplicates: 440
Category
Breakfast               122
Beverages               107
Baking & Flour           48
Snacks                   47
Cleaning-supplies        22
Home & kitchen           16
Personal-care            14
Baby care                13
Dairy & Eggs             12
Canned food               8
Meat & Fish               7
Spices & Ingredients      7
Oil                       6
Sauces & pickles          6
Salt-sugar                2
Frozen Snacks             2
Candy & chocolate         1
Name: count, dtype: int64


We found many duplicates across categories in the df_chaldal dataframe, which may bias comparisons if not removed. No cross-category duplicates were found in df_shwapno.

**step-3:** checking of duplicate Product Titles Across Different Categories:

In [23]:
pd.set_option('display.width', 1000)    

result = subcat_duplicates_chaldal.groupby('Title')['Category'].apply(lambda x: ' , '.join(x.unique())).reset_index()
result.columns = ['Title', 'Found in Categories']
result

,Title,Found in Categories
0,AMA Coffee 3 In 1,"Breakfast , Beverages"
1,Ahmed White Vinegar,"Snacks , Baking & Flour"
2,Almarai Spreadable Cream Cheese,"Dairy & Eggs , Breakfast"
3,Almarai spreadable Cream Cheese,"Dairy & Eggs , Breakfast"
4,Almonds (Kath Badam),"Snacks , Baking & Flour"
...,...,...
186,Women's Plus Horlicks Health And Nutrition Dri...,"Breakfast , Beverages"
187,Xpel Kids Spray & Lotion Twin Set,"Personal-care , Cleaning-supplies"
188,Yage Electronic Mosquito Swatter (White & Blue),"Cleaning-supplies , Home & kitchen"
189,Yage Electronic Mosquito Swatter (White & Pink),"Cleaning-supplies , Home & kitchen"


**step-4:** Here, We used category-specific rules (keywords) to decide which ones are real duplicates and removed them.

In [24]:
import re

def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text

cols_to_clean = ['Title', 'Category', 'Unit']

for col in cols_to_clean:
    if col in df_chaldal.columns:
        df_chaldal[col] = df_chaldal[col].apply(normalize_text)


subcat_duplicates_chaldal = df_chaldal[df_chaldal.duplicated(subset=['Title', 'Unit', 'Previous_price', 'Current_price'], keep=False)].copy()

rules = {
    'Beverages' : ['tea', 'coffee', 'nescafe', 'cha', 'horlicks', 'drink', 'syrup', 'complan', 'boost', 'infusion', 'maccoffee'],
    'oil' : ['oil'],
    'dairy & eggs' : ['cheese', 'milk'],
    'baking & flour' : ['vinegar', 'custard', 'flour', 'butter', 'whip topping'],
    'snacks' : ['badam', 'nuts', 'almond', 'cashews', 'rice', 'pistachios', 'peanuts'],
    'frozen snacks' : ['paratha', 'walnuts'],
    'sauces & pickles' : ['sauce', 'balachaw'],
    'breakfast': ['mayonnaise'],
    'cleaning-supplies' : ['mosquito', 'cover', 'scourer', 'scrubber', 'gloves', 'bag', 'hand towel', 'spray', 'tissue'],
    'home & kitchen' : ['holder', 'kitchen', 'rack'],
    'canned food' : ['tuna', 'gherkins'],
    'personal-care' : ['facial', 'paper napkins'],
    'Baby care' : ['baby'],
    'salt-sugar' : ['sugar'],
    'candy & chocolate' : ['chocolate']
}

indices_to_drop = set() 

for category, keywords in rules.items():
    pattern = '|'.join([rf'\b{k}s?\b' for k in keywords])

    mask = (
        (subcat_duplicates_chaldal['Category'] != category) & 
        (subcat_duplicates_chaldal['Title'].str.contains(pattern, na=False, regex=True)) 
    )
    
    target_indices = subcat_duplicates_chaldal[mask].index.tolist()
    indices_to_drop.update(target_indices)


df_chaldal = df_chaldal.drop(list(indices_to_drop)).reset_index(drop=True)

print(f"Total removed duplicates: {len(indices_to_drop)}")


Total removed duplicates: 350


**step-5:** Remaining duplicate check:

In [25]:
subcat_duplicates_chaldal = df_chaldal[df_chaldal.duplicated(subset=['Title', 'Unit', 'Previous_price', 'Current_price'], keep=False)].copy()

result = subcat_duplicates_chaldal.groupby('Title')['Category'].apply(lambda x: ' , '.join(x.unique())).reset_index()
result.columns = ['Title', 'Found in Categories']

print(result)

Empty DataFrame
Columns: [Title, Found in Categories]
Index: []


**f. Final checking:**

In [26]:
df_chaldal.info()

<class 'pandas.DataFrame'>
RangeIndex: 3366 entries, 0 to 3365
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Category        3366 non-null   str  
 1   Title           3366 non-null   str  
 2   Currency        3366 non-null   str  
 3   Previous_price  3366 non-null   int64
 4   Current_price   3366 non-null   int64
 5   Unit            3366 non-null   str  
dtypes: int64(2), str(4)
memory usage: 157.9 KB


### **3. Combine shwapno and chaldal dataframe:**

**a. combine two dataframe:**

In [27]:
df_shwapno['Source'] = 'Shwapno'
df_chaldal['Source'] = 'Chaldal'

df_combined = pd.concat([df_shwapno, df_chaldal], ignore_index=True)

In [28]:
print(df_combined['Source'].value_counts())
len(df_combined)

Source
Shwapno    3874
Chaldal    3366
Name: count, dtype: int64


7240

**b. Data stucture:**

In [29]:
df_combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 7240 entries, 0 to 7239
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Category        7240 non-null   str    
 1   Title           7240 non-null   str    
 2   Currency        7240 non-null   str    
 3   Previous_price  7240 non-null   float64
 4   Current_price   7240 non-null   float64
 5   Unit            7240 non-null   str    
 6   Source          7240 non-null   str    
dtypes: float64(2), str(5)
memory usage: 396.1 KB


**c. Handling Null :** 

**step-1: Cleaning fake data, then null check :**

In [30]:
import numpy as np

df_combined = df_combined.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
df_combined.replace(['Nan', 'None', '', 'nan', 'null',  r'^\s*$'], np.nan, regex=True, inplace=True)

df_combined.isnull().sum()

Category           0
Title             43
Currency           0
Previous_price     0
Current_price      0
Unit              88
Source             0
dtype: int64

Here, Identified missing values in `Title` and `Unit` columns. Here, We drop rows with missing titles.

**step-2: Check title column & drop rows:**

In [31]:
df_combined[df_combined['Title'].isnull()].head(5)

,Category,Title,Currency,Previous_price,Current_price,Unit,Source
0,Fruits,NaN,৳,13.00,13.00,Per Piece,Shwapno
1,Fruits,NaN,৳,5.75,5.75,Per Piece,Shwapno
47,Vegetables,NaN,৳,12.00,12.00,Per Piece,Shwapno
71,Vegetables,NaN,৳,40.00,40.00,Per Piece,Shwapno
83,Vegetables,NaN,৳,30.00,30.00,Per Piece,Shwapno


In [32]:
df_combined = df_combined.dropna(subset=['Title']).reset_index(drop=True)

In [33]:
df_combined.isnull().sum()

Category           0
Title              0
Currency           0
Previous_price     0
Current_price      0
Unit              88
Source             0
dtype: int64

**d. Here, extract 'Brand' from the title :**

**Step-1 :** Creating `brand column` to find Brand name and make a `CSV file` to extract brand names for all categories :

In [34]:
from collections import Counter

def text_clean(text):
    if pd.isna(text) or str(text).strip() == "":
        return ""
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", ' ', text) 
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def brand_name_search(df_combined, col_name, filename='combined_data_BrandName_3.csv'):
  
    csv_df = df_combined.copy()
    csv_df['cleaned_title'] = csv_df[col_name].apply(text_clean)
    
    pattern_3_map = {} 
    words_1, words_2, words_3 = [], [], []

    for _, row in csv_df.iterrows():
        text = row['cleaned_title']
        words = text.split()
        if not words: continue
            
    
        w1 = words[0]
        words_1.append(w1)
        
        w2 = " ".join(words[:2]) if len(words) >= 2 else w1
        words_2.append(w2)
        
        if len(words) >= 3:
            w3 = " ".join(words[:3])
        else:
            w3 = " ".join(words) 

        words_3.append(w3)
        
        if w3 not in pattern_3_map:
            pattern_3_map[w3] = []
        pattern_3_map[w3].append({'cat': row['Category'], 'ttl': row[col_name]})

    count_1 = Counter(words_1)
    count_2 = Counter(words_2)
    count_3 = Counter(words_3)

    final_data = []
    last_w1, last_w2, last_w3 = None, None, None

  
    for w1, c1 in sorted(count_1.items(), key=lambda x: x[1], reverse=True):
        matched_w2 = sorted([p for p in count_2 if p.startswith(f"{w1} ") or p == w1], 
                            key=lambda x: count_2[x], reverse=True)
        for w2 in matched_w2:
            c2 = count_2.get(w2, "")
            matched_w3 = sorted([p for p in count_3 if p.startswith(f"{w2} ") or p == w2], 
                                key=lambda x: count_3[x], reverse=True)
            for w3 in matched_w3:
                c3 = count_3.get(w3, "")
                titles = pattern_3_map.get(w3, [])
                for item in titles:
                  
                    display_w1 = w1 if w1 != last_w1 else ""
                    display_c1 = c1 if w1 != last_w1 else ""
                    display_w2 = w2 if w2 != last_w2 else ""
                    display_c2 = c2 if w2 != last_w2 else ""
                    display_w3 = w3 if w3 != last_w3 else ""
                    display_c3 = c3 if w3 != last_w3 else ""

                    final_data.append({
                        'Category': item['cat'],
                        'Brand_name': display_w1,
                        'first_1_count': display_c1,
                        'first_2_word': display_w2,
                        'first_2_count': display_c2,
                        'first_3_word': display_w3,
                        'first_3_count': display_c3,
                        'Original_Title': item['ttl']
                    })
                    last_w1, last_w2, last_w3 = w1, w2, w3


    result_df = pd.DataFrame(final_data)
    result_df.to_csv(filename, index=False, encoding='utf-8-sig')

    print(f"Total number of unique word-1: {len(count_1)}")
    print(f"Total number of unique word-2: {len(count_2)}")
    print(f"Total number of unique word-3: {len(count_3)}")
    print(f"CSV created: {filename}")
    
    return result_df


full_brand_data = brand_name_search(df_combined, 'Title')

Total number of unique word-1: 1181
Total number of unique word-2: 3495
Total number of unique word-3: 4769
CSV created: combined_data_BrandName_3.csv


**Step - 2:** combined_data_BrandName_3.csv was processed into combined_data_BrandName_cleaned_3.csv by creating three reference columns—Brand_name, Fresh_produce, and Non_brand—and matching product titles to assign the appropriate category :

In [87]:
import re

def text_clean(text):
    if pd.isna(text) or str(text).strip() == "":
        return ""
    
    text = str(text).lower()
    text = re.sub(r"['’]?s(?:\s|$)", ' ', text) 
    text = re.sub(r'\band\b', '&', text)
    text = re.sub(r"[^\w\s&]", ' ', text) 
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def apply_final_branding(df_combined, col_name, brand_ref_csv):
    try:
        ref_df = pd.read_csv(brand_ref_csv, encoding='utf-8-sig')
    except Exception:
        ref_df = pd.read_csv(brand_ref_csv, encoding='latin1')
    
    mapping_dict = {
        'Fair & Lovely': 'Glow & Lovely',
        'wonder' : 'wonder-aci',
        'muffin' : 'wonder-pran',
        'cadbary' : 'cadbury',
        'moccana' : 'moccona',
        'parodntax' : 'parodontax',
        'skin cafe' : 'skin cafã',
        'mr twist': 'Bombay',
        'mr pasta': 'Bombay',
        'earth b&y' : 'Earth Beauty & You',
    }
    
    fresh_produce_list = []
    if 'Fresh_Produce' in ref_df.columns:
        fresh_produce_list = [text_clean(f) for f in ref_df['Fresh_Produce'].dropna().unique() if text_clean(f) != ""]

    non_brand_list = []
    if 'Non_Branded' in ref_df.columns:
        non_brand_list = [text_clean(u) for u in ref_df['Non_Branded'].dropna().unique() if text_clean(u) != ""]
    
    brand_data = []
    if 'Brand_name' in ref_df.columns:
        for b in ref_df['Brand_name'].dropna().unique():
            b_str = str(b).strip()
            cleaned = text_clean(b_str)
            if cleaned:
                brand_data.append({
                    'original': b_str,
                    'clean': cleaned,
                    'no_space': cleaned.replace(" ", "")
                })
    
    brand_data = sorted(brand_data, key=lambda x: len(x['clean']), reverse=True)

    def get_brand(row):
        title_raw = str(row[col_name])
        title_cleaned = text_clean(title_raw)
        if not title_cleaned: return "Pending"
        
        if title_cleaned.startswith(('shwapno', 'chaldal')):
            return "House Brand"

        if fresh_produce_list and title_cleaned.startswith(tuple(fresh_produce_list)):
            return "Fresh Produce"
        
        if non_brand_list and title_cleaned.startswith(tuple(non_brand_list)):
            return "Non-Brand"

        title_no_spaces = title_cleaned.replace(" ", "")
        for b in brand_data:
            if title_cleaned.startswith(b['clean']) or \
               title_no_spaces.startswith(b['no_space']) or \
               (f" {b['clean']}" in f" {title_cleaned}"): 
                
                res = b['original']
                return mapping_dict.get(res, res)

        return "Pending"

    df_combined['Brand'] = df_combined.apply(get_brand, axis=1)

    brand_counts = df_combined['Brand'].value_counts().reset_index()
    brand_counts.columns = ['Brand Name', 'Total Count']

    actual_unique_brands = df_combined[~df_combined['Brand'].isin(['Non Brand', 'Fresh Produce', 'House Brand', 'Pending'])]['Brand'].nunique()
    unique_pending_count = df_combined[df_combined['Brand'] == 'Pending'][col_name].nunique()

    print(f"Total number of unique Brands : {actual_unique_brands}")
    print(f"Total number of pending Brands : {unique_pending_count}")
    print("\n--- Brand Summary ---")
    print(brand_counts)
    
    return df_combined, brand_counts

df_combined, stats = apply_final_branding(df_combined, 'Title', '../data/interim/combined_data_BrandName_cleaned_3.csv')

Total number of unique Brands : 704
Total number of pending Brands : 0

--- Brand Summary ---
        Brand Name  Total Count
0        Non-Brand          955
1    Fresh Produce          479
2            fresh          182
3      House Brand          171
4             pran          156
..             ...          ...
701        citizen            1
702         alteco            1
703        a4 tech            1
704         montex            1
705        century            1

[706 rows x 2 columns]


**e. Check column name & Rename :**

In [36]:
print(df_combined.columns)

Index(['Category', 'Title', 'Currency', 'Previous_price', 'Current_price', 'Unit', 'Source', 'Brand'], dtype='str')


In [37]:
df_combined.rename(columns={
    "Category" : "Sub-category",
}, inplace=True)

print(df_combined.columns)

Index(['Sub-category', 'Title', 'Currency', 'Previous_price', 'Current_price', 'Unit', 'Source', 'Brand'], dtype='str')


**f. Print all Sub-categories unique names:** 

In [38]:
df_combined['Sub-category'] = df_combined['Sub-category'] = df_combined['Sub-category'].str.title() 
df_combined['Sub-category'].unique()

<StringArray>
['Fruits', 'Vegetables', 'Meat & Fish', 'Dairy & Eggs', 'Rice', 'Oil', 'Lentils & Pulses', 'Salt & Sugar', 'Spices & Ingredients', 'Sauces & Pickles', 'Breakfast', 'Snacks', 'Candy, Chocolate & Ice-Cream', 'Beverages', 'Baking & Flour', 'Frozen Snacks', 'Canned Food', 'Personal Care', 'Baby Care', 'Cleaning-Supplies', 'Home & Kitchen', 'Stationeries', 'Salt-Sugar', 'Candy & Chocolate', 'Personal-Care']
Length: 25, dtype: str

**g. sub-categories mapping:**

In [39]:
category_map = {
     
    'Candy & Chocolate': 'Chocolates & Icecream',
    'Candy, Chocolate & Ice-Cream' : 'Chocolates & Icecream',
    'Personal-Care' : 'Personal Care',
    'Salt-Sugar' : 'Salt & Sugar'
}

df_combined['Sub-category'] = df_combined['Sub-category'].replace(category_map)
df_combined['Sub-category'].unique()

<StringArray>
['Fruits', 'Vegetables', 'Meat & Fish', 'Dairy & Eggs', 'Rice', 'Oil', 'Lentils & Pulses', 'Salt & Sugar', 'Spices & Ingredients', 'Sauces & Pickles', 'Breakfast', 'Snacks', 'Chocolates & Icecream', 'Beverages', 'Baking & Flour', 'Frozen Snacks', 'Canned Food', 'Personal Care', 'Baby Care', 'Cleaning-Supplies', 'Home & Kitchen', 'Stationeries']
Length: 22, dtype: str

**h. Here, we mapping sub-categories into five mejor categories:**

In [40]:
category_maping = {
  'Vegetables' : 'Fresh & Perishables',
  'Fruits' : 'Fresh & Perishables',
  'Meat & Fish' : 'Fresh & Perishables',
  'Dairy & Eggs' : 'Fresh & Perishables',

  'Rice' : 'Cooking Essentials & Staples',
  'Lentils & Pulses' : 'Cooking Essentials & Staples',
  'Oil' : 'Cooking Essentials & Staples',
  'Salt & Sugar' : 'Cooking Essentials & Staples',
  'Spices & Ingredients' : 'Cooking Essentials & Staples',
  'Baking & Flour' : 'Cooking Essentials & Staples',
 
  'Snacks' : 'Beverages & Instant Food',
  'Breakfast' : 'Beverages & Instant Food',
  'Beverages' : 'Beverages & Instant Food',
  'Sauces & Pickles' : 'Beverages & Instant Food',
  'Chocolates & Icecream' : 'Beverages & Instant Food',
  'Frozen Snacks' : 'Beverages & Instant Food',
  'Canned Food' : 'Beverages & Instant Food',
  
  'Personal Care' : 'Personal & Baby care',
  'Baby Care' : 'Personal & Baby care',

  'Cleaning-Supplies' : 'Home & Lifestyle',
  'Home & Kitchen' : 'Home & Lifestyle',
  'Stationeries' : 'Home & Lifestyle'
}

df_combined['Category'] = df_combined['Sub-category'].map(category_maping)

cols = ['Category'] + [col for col in df_combined.columns if col != 'Category']
df_combined = df_combined[cols]

print(df_combined['Category'].value_counts())

df_combined.columns

Category
Personal & Baby care            2626
Beverages & Instant Food        2140
Cooking Essentials & Staples     927
Home & Lifestyle                 845
Fresh & Perishables              659
Name: count, dtype: int64


Index(['Category', 'Sub-category', 'Title', 'Currency', 'Previous_price', 'Current_price', 'Unit', 'Source', 'Brand'], dtype='str')

**i. Section-wise mapping: Food items & Non-food items:**

In [41]:
section_mapping = {
    'Fresh & Perishables': 'Food Items',
    'Cooking Essentials & Staples': 'Food Items',
    'Beverages & Instant Food': 'Food Items',
    'Home & Lifestyle': 'Non-Food Items',
    'Personal & Baby care': 'Non-Food Items',
}

df_combined['Section'] = df_combined['Category'].map(section_mapping)

cols = ['Section'] + [col for col in df_combined.columns if col != 'Section']
df_combined = df_combined[cols]

print(df_combined['Section'].value_counts())

Section
Food Items        3726
Non-Food Items    3471
Name: count, dtype: int64


**j. In this step, clean the `Title` and split it into `product_name` and `Extra_info` column to obtain `accurate unit` and `offer` information for analysis:**

In [42]:
clean_title = df_combined['Title'].str.lower().str.strip().str.replace(r'\s+', ' ', regex=True)

df_combined[['Product_name', 'Extra_info']] = clean_title.str.extract(
    r'^((?:.*?\b\d+in\d+\b|.*?))\s*(?=' 
    r'(?:'
    r'\bbuy\s*&?\s*get\b|'
    r'\bbuy\s*\d+|'
    r'\bsave\b|'
    r'\bfree\b|'
    r'\(?\s*(?<!in)\d+.*?' 
    r'(?:kg|gm|ltr|ml|pcs|each|l|g|rim|capsules|tablets|cc|m|mtr|box|pack?|set|pair?|watt|staples|sachets?|pads?|pages?)'
    r'(?:\s*pack)?\s*\)?'
    r'))\s*(.*)$',
    flags=re.IGNORECASE
)
df_combined['Product_name'] = df_combined['Product_name'].fillna(df_combined['Title'])

**k. Standardize the `unit column` :**

**Step-1 :** Unit Frequency Analysis:

In [43]:
unit_only_counts = (
    df_combined['Unit']
    .str.lower()
    .str.replace(r'[\d\.]+', '', regex=True)
    .str.strip() 
)

print(unit_only_counts.value_counts(dropna=False))

Unit
per piece    3642
gm           1418
ml            856
each          391
pcs           270
kg            193
per kg        164
ltr           144
NaN            88
pack            9
bundle          8
per pack        7
per unit        5
bundles         1
tablets         1
Name: count, dtype: int64


**Step-3 :** Unit Normalization :

In [44]:
combined_unit_clear = (
    df_combined['Unit']
    .str.lower()                                  
    .str.replace(r'\bper\b', '', regex=True)     
    .str.replace(r'\s+', ' ', regex=True)       
    .str.strip()
)

print(combined_unit_clear.str.extract(r'([a-z]+)')[0].value_counts(dropna=False))

0
piece      3642
gm         1418
ml          856
each        391
kg          357
pcs         270
ltr         144
NaN          88
pack         16
bundle        8
unit          5
bundles       1
tablets       1
Name: count, dtype: int64


**Step-4:** Unit Extraction and Data ImputationIn :  Extracting hidden weight and count data from `Extra_info` column and metadata to resolve missing values and ensure unit consistency :

In [45]:
measure_units = ['kg', 'g', 'gm', 'gram', 'ml', 'ltr', 'lit', 'liter', 'mg', 'cc', 'mtr', 'm', 'cm']
count_units = ['pcs', 'packets', 'units', 'tabs', 'bundle', 'bundles', 'capsules', 'tablets', 'pads', 'box', 'set', 'pair', 'pairs', 'staples', 'page', 'pages', 'rim']

def extract_unit_pattern(text):
    if pd.isna(text) or str(text).strip() == "":
        return ""
    
    text_str = str(text).strip().lower()
    all_units = measure_units + count_units
    unit_pattern = '|'.join(all_units)

    stop_words = ['buy', 'get', 'free']
    should_skip_sum = any(word in text_str for word in stop_words)

    parts = re.split(r'\b(get|save|free|free/)\b', text_str)
    search_text = parts[0].strip() 
  
    weight_p = '|'.join(measure_units)
    range_match = re.findall(rf'(\d+\.?\d*)\s*-\s*(\d+\.?\d*)\s*({weight_p})', search_text)
    if range_match:
        num1, num2, unit = range_match[0]
        avg = (float(num1) + float(num2)) / 2
        return f"{int(avg) if avg.is_integer() else avg}{unit}"

    pm_pattern = rf'(\d+\.?\d*)\s*[\(]?\s*[â±±]+\s*[\)]?\s*\d+\.?\d*\s*({unit_pattern})\b'
    pm_match = re.search(pm_pattern, search_text)
    if pm_match:
        return f"{pm_match.group(1)}{pm_match.group(2)}"

    pattern = rf'(\d+\.?\d*)\s*({unit_pattern})\b'
    matches = re.findall(pattern, search_text)
    
    if matches:
        m_matches = [m for m in matches if m[1] in measure_units]
        
        if len(m_matches) > 1 and not should_skip_sum:
            total_val = 0.0
            big_units = ['ltr', 'lit', 'liter', 'kg']
            has_big_unit = any(m[1] in big_units for m in m_matches)
            is_liquid = any(m[1] in ['ml', 'ltr', 'lit', 'liter', 'cc'] for m in m_matches)
            
            for val, unit in m_matches:
                v = float(val)
                if unit in big_units:
                    total_val += v * 1000
                elif unit == 'mg':
                    total_val += v / 1000
                else:
                    total_val += v
            
            if has_big_unit or total_val >= 1000:
                final_val = total_val / 1000
                u = 'ltr' if is_liquid else 'kg'
                return f"{int(final_val) if final_val.is_integer() else round(final_val, 3)}{u}"
            else:
                return f"{int(total_val) if total_val.is_integer() else total_val}{m_matches[0][1]}"
        
        if m_matches:
            return f"{m_matches[0][0]}{m_matches[0][1]}"
            
        return f"{matches[0][0]}{matches[0][1]}"
                 
    return ""

def get_actual_unit(row):
    unit_raw = str(row['Unit']).strip().lower() if pd.notna(row['Unit']) else ""
    extra_info = str(row['Extra_info']).strip() if pd.notna(row['Extra_info']) else ""
    
    count_keywords = count_units + ['pack', 'unit', 'each', 'piece']

    if any(word in unit_raw for word in measure_units):
        res = extract_unit_pattern(unit_raw)
        return res if res else unit_raw

    elif any(word in unit_raw for word in count_keywords) or unit_raw == "":
        
        res_extra = extract_unit_pattern(extra_info)
        if res_extra:
            return res_extra
        
        return unit_raw

    return ""

df_combined['Actual_unit'] = df_combined.apply(get_actual_unit, axis=1)

print(f"Total stored: {(df_combined['Actual_unit'] != '').sum()}")

Total stored: 7192


**Step-5:** Checking null of `Actual_unit` :

In [46]:
selected_columns = ["Title", "Unit", "Actual_unit", "Previous_price", "Current_price"]

null_unit_view = df_combined.loc[df_combined["Actual_unit"].isin(['', None]), selected_columns]

null_unit_view

,Title,Unit,Actual_unit,Previous_price,Current_price
5685,veet expert all-in-one women's trimmer,NaN,,3200.0,3200.0
6960,transtec green ww led bulb (screw) 15 watt,NaN,,400.0,400.0
6961,transtec green cdl led bulb (pin) 15 watt,NaN,,400.0,400.0
6984,portable digital weight scale,NaN,,490.0,490.0
7088,deli auto numbering machine 6 digits,NaN,,969.0,969.0


**Step-6:** Assigned `1pcs` to discrete items (like tools/electronics) missing a unit to ensure data consistency :

In [47]:
target_keywords = ["women's trimmer", 'weight scale', 'numbering machine', 'bulb']

for keyword in target_keywords:
    matched_indices = null_unit_view[null_unit_view["Title"].str.lower().str.contains(keyword, na=False)].index
    
    df_combined.loc[matched_indices, "Actual_unit"] = "1pcs"

In [48]:
selected_columns = ["Title", "Unit", "Actual_unit", "Previous_price", "Current_price"]

null_unit_view = df_combined.loc[df_combined["Actual_unit"].isin(['', None]), selected_columns]
null_unit_view

,Title,Unit,Actual_unit,Previous_price,Current_price


**l. Creating another `Offer_Status` and convert it into value for checking promotional offers:**

**Step-1:** Create `Offer_staus` column:

In [49]:

def check_offer(title):
    title_str = str(title)
    title_lower = title_str.lower()
    
    ignore_words = ['sugar free', 'pin', 'pads']

    bracket_offer_match = re.search(r'\(([^)]*(?:free|save|get|dooper)[^)]*)\)', title_lower)
    
    if ('buy' in title_lower and 'get' in title_lower) or \
       'buy' in title_lower or \
       'save' in title_lower or \
       bracket_offer_match:
        
        all_brackets = re.findall(r'\((.*)\)', title_str)
        if all_brackets:
       
            for content in reversed(all_brackets):
                c_low = content.lower()
                
  
                if any(ignore in c_low for ignore in ignore_words):
                    continue
                
      
                if any(word in c_low for word in ['free', 'save', 'get', 'dooper', 'buy']):
                    return content.strip()
            return 'No Offer' 
        
        return 'No Offer'
    else:
        return 'No Offer'

df_combined['Offer_Status'] = df_combined['Title'].apply(check_offer)

print("--- Offer Status Counts ---")
print(df_combined['Offer_Status'].value_counts())

--- Offer Status Counts ---
Offer_Status
No Offer                      6974
Buy2 Get1 Free                  51
Buy3 Get1 Free                  28
Buy1 Get1 Free                  25
buy 2 get 1 free                 6
                              ... 
free 75 gm baby soap             1
free detergent                   1
free 100 ml                      1
free 100 ml + scrubber           1
free mine detergent 200 gm       1
Name: count, Length: 99, dtype: int64


**Step-2:** Create `Actual_unit_price` and `Savings_amount` column from Offer_staus column info:

In [50]:
def advanced_offer_calc(row):
    offer = str(row['Offer_Status']).lower()
    
    try:
        price = float(row['Current_price'])
    except:
        return None, 0
    
    actual_unit_str = str(row['Actual_unit']).lower()
    a_unit_num_match = re.search(r'(\d+(?:\.\d+)?)', actual_unit_str)
    a_unit_name_match = re.search(r'([a-zA-Z]+)', actual_unit_str)
    
    actual_val = float(a_unit_num_match.group(1)) if a_unit_num_match else 0
    actual_unit_name = a_unit_name_match.group(1) if a_unit_name_match else ""
    

    if actual_unit_name in ['kg', 'ltr', 'lit', 'liter']:
        actual_val_converted = actual_val * 1000
    else:
        actual_val_converted = actual_val
    
    eff_price = None
    savings = 0.0

    save_match = re.search(r'(?:save|saving|savings)\s*(?:tk\.?|rs\.?|৳)?\s*(\d+(?:\.\d+)?)', offer)
    if save_match:
        savings = float(save_match.group(1))

    all_units = measure_units + count_units
    units_pattern = '|'.join(all_units)
    
    get_complex_match = re.search(rf'get\s*(\d+(?:\.\d+)?).*?({units_pattern})\b', offer)

    if not get_complex_match and savings == 0:
        three_digit_get = re.search(r'get\s*(\d{3,})', offer)
        if three_digit_get and actual_val_converted > 0:
            free_val = float(three_digit_get.group(1))
            unit_rate = price / actual_val_converted
            savings = unit_rate * free_val


    buy_match = re.search(r'buy\s*(\d+)(?!\s*(?:gm|ml|kg|pcs|unit))', offer)
    get_only_match = re.search(r'get\s*(\d+)(?!\s*(?:gm|ml|kg|pcs|unit))', offer)

    if buy_match:
        eff_price = price / float(buy_match.group(1))
    elif get_only_match and not get_complex_match and savings == 0:
        eff_price = price / float(get_only_match.group(1))

    if get_complex_match and actual_val_converted > 0:
        free_val = float(get_complex_match.group(1))
        free_unit_name = get_complex_match.group(2)
        
        if free_unit_name in ['kg', 'ltr', 'lit', 'liter']:
            free_val_converted = free_val * 1000
        else:
            free_val_converted = free_val
            
        unit_rate = price / actual_val_converted 
        savings = unit_rate * free_val_converted

    if buy_match and get_only_match and savings == 0:
        buy_q = float(buy_match.group(1))
        get_q = float(get_only_match.group(1))
        temp_eff_price = price / buy_q
        savings = temp_eff_price * get_q

    if eff_price is not None:
        if round(eff_price, 2) == round(price, 2):
            eff_price = None
        else:
            eff_price = round(eff_price, 2)
            
    return eff_price, round(savings, 2)

df_combined[['Actual_unit_price', 'Savings_Amount']] = df_combined.apply(
    lambda row: pd.Series(advanced_offer_calc(row)), axis=1
)


**Step-3:** Fillup Null cells of `Actual_unit_price` from `Current_price` column:

In [51]:
df_combined['Actual_unit_price'] = (df_combined['Actual_unit_price'].replace('', np.nan).fillna(df_combined['Current_price']))
df_combined['Actual_unit_price'] = pd.to_numeric(df_combined['Actual_unit_price'],errors='coerce')

**Step-4:** Create `Discounted_amount` column :

In [52]:
df_combined['Discounted_amount'] = df_combined['Previous_price'] - df_combined['Current_price']

**Step-5:** Create `Total_savings` & `Free_product` column :

In [53]:
savings_amount = pd.to_numeric(df_combined['Savings_Amount'], errors='coerce').replace(0, np.nan)
savings_count = savings_amount.notna().sum()

disc_val = pd.to_numeric(df_combined['Discounted_amount'], errors='coerce')

discount_fill_mask = (
    savings_amount.isna() &
    pd.notna(disc_val) &
    (disc_val > 0)
)

discount_fill_count = discount_fill_mask.sum()

savings_amount.loc[discount_fill_mask] = disc_val

df_combined['Total_savings'] = savings_amount

df_combined['Free_product'] = pd.Series([np.nan] * len(df_combined), dtype='object')

status_clean = (df_combined['Offer_Status'].fillna('').astype(str).str.strip().str.lower())

free_product_mask = (
    savings_amount.isna() &
    status_clean.ne('no offer')
)

df_combined.loc[free_product_mask, 'Free_product'] = 'Product free'

free_product_count = free_product_mask.sum()

print(f"Total_savings status: Savings fill count = {savings_count} | Discount fill count = {discount_fill_count}")
print(f"Free_product status: Product free count = {free_product_count}")

Total_savings status: Savings fill count = 193 | Discount fill count = 1019
Free_product status: Product free count = 30


**Step-6:** Create `Savings_products_status` column:

In [54]:
conditions = [
    (df_combined['Total_savings'].fillna(0) > 0) & (df_combined['Free_product'].notnull()) & (df_combined['Free_product'] != ""),
    (df_combined['Total_savings'].fillna(0) > 0),
    (df_combined['Free_product'].notnull()) & (df_combined['Free_product'] != ""),
]

choices = [
    "Savings&product both",
    "Money Savings",
    "Free product",
]

df_combined['Savings_freeProduct_status'] = np.select(conditions, choices, default="")

counts = df_combined['Savings_freeProduct_status'].value_counts()
print(counts)

Savings_freeProduct_status
                 5955
Money Savings    1212
Free product       30
Name: count, dtype: int64


**m. Rename Current_price column to Sales_price :**

In [55]:

df_combined.rename(columns={'Current_price' : 'Sales_price'}, inplace=True)

**n. Create `Market_price` column:**

In [56]:
df_combined['Market_price'] = df_combined['Sales_price'] + df_combined['Total_savings']
df_combined['Market_price'] = pd.to_numeric(df_combined['Market_price'],errors='coerce')

**o. Structural Decomposition of Actual_unit column :** Converted unstructured unit formats into consistent features like  `Measure_value`, `Measure_unit`, `Count_value` and `Count_unit` for easier comparison and downstream analytics:

**Step-1:** Unique Units Count:

In [57]:
df_combined['Standard_unit'] = (df_combined['Actual_unit'].str.lower().str.strip())

only_text_units = (df_combined['Standard_unit'].str.replace(r'[\d\.]+', '', regex=True).str.strip()                             )
clean_unit_counts = only_text_units.value_counts(dropna=False)

print(clean_unit_counts)

Standard_unit
gm           3399
ml           2095
kg            516
each          326
ltr           265
pcs           250
per piece     184
pads           43
g              37
cm             27
m              10
bundle          8
rim             8
pack            7
cc              5
mtr             2
box             2
set             2
pair            2
pages           2
page            2
capsules        1
bundles         1
mg              1
pairs           1
staples         1
Name: count, dtype: int64


**Step-3:** Standard_unit mapping:

In [58]:
df_combined['Standard_unit'] = df_combined['Standard_unit'].str.replace(r'\bper piece\b', '1 pcs', regex=True, case=False).str.strip()

extracted_val = df_combined['Standard_unit'].str.extract(r'(\d*\.?\d+)')[0]
raw_unit = df_combined['Standard_unit'].str.extract(r'([a-zA-Z]+)')[0].str.lower()

extracted_val = extracted_val.fillna(raw_unit.apply(lambda x: '1' if pd.notnull(x) else ''))

unit_map = {
    'g': 'gm', 'm': 'mtr', 'cc': 'ml', 'each': 'pcs',
    'pads': 'pack', 'capsules': 'pcs', 'staples': 'pcs', 'box': 'pack',
    'bundles': 'bundle', 'pairs': 'pair', 'pages': 'page', 'set': 'pcs', 'rim': 'pack'
}
mapped_unit = raw_unit.replace(unit_map)

df_combined['Standard_unit'] = (
    extracted_val + " " + mapped_unit.fillna('')
).str.strip()


only_text_units = (df_combined['Standard_unit'].str.replace(r'[\d\.]+', '', regex=True).str.strip()                             )
clean_unit_counts = only_text_units.value_counts(dropna=False)

print(clean_unit_counts)

Standard_unit
gm        3436
ml        2100
pcs        764
kg         516
ltr        265
pack        60
cm          27
mtr         12
bundle       9
page         4
pair         3
mg           1
Name: count, dtype: int64


**Step-4:**

In [59]:
result = pd.crosstab(
    df_combined['Sub-category'], 
    df_combined['Standard_unit'].str.extract(r'([a-zA-Z]+)')[0].str.lower()
)
target_units = ['gm', 'kg', 'ml', 'ltr', 'g', 'cm', 'm', 'cc', 'mtr', 'mg', 'each', 'pcs', 'piece', 'pads', 'pack', 'bundle', 'rim', 'box', 'set', 'pair', 'page', 'pages', 'capsules', 'bundles', 'pairs', 'staples']

existing_targets = [u for u in target_units if u in result.columns]
print(result[existing_targets])

0                       gm   kg    ml  ltr  cm  mtr  mg  pcs  pack  bundle  pair  page
Sub-category                                                                          
Baby Care              139   51   112    1   0    3   0   84     0       0     0     0
Baking & Flour         105   36    34    0   1    1   0   13     0       0     0     0
Beverages              187    7   215   81   0    0   0    7     0       0     0     0
Breakfast              116   10    17    0   0    0   0    1     0       0     0     0
Canned Food             52    0     1    0   0    0   0    0     0       0     0     0
Chocolates & Icecream  199    0    21   13   0    0   0    6     0       0     0     0
Cleaning-Supplies      108   28   203   36   1    0   0   77     0       0     1     0
Dairy & Eggs           145   29    52   15   0    0   0   18     0       0     0     0
Frozen Snacks          169    2     0    0   0    0   0    0     0       0     0     0
Fruits                   8   26     0    0 

**Step-4:** Convert to standard formet of unit of `Standard_unit` columns:

In [60]:
weight_to_kg = {
    'gm': 0.001,
    'mg': 0.000001,
    'kg': 1
}

volume_to_ltr = {
    'ml': 0.001,
    'ltr': 1,
}

length_to_mtr = {
    'cm': 0.01,
    'mtr': 1,
}

def convert_unit(text):
    text = str(text).lower()

    num_match = re.search(r'(\d*\.?\d+)', text)
    if not num_match:
        return text 

    value = float(num_match.group())

    unit_match = re.search(r'[a-zA-Z]+', text)
    if not unit_match:
        return text

    unit = unit_match.group().lower()

    if unit in weight_to_kg:
        converted_value = value * weight_to_kg[unit]
        return f"{round(converted_value, 4)} kg"

    elif unit in volume_to_ltr:
        converted_value = value * volume_to_ltr[unit]
        return f"{round(converted_value, 4)} ltr"

    elif unit in length_to_mtr:
        converted_value = value * length_to_mtr[unit]
        return f"{round(converted_value, 4)} mtr"

    elif unit == 'pair':
        converted_value = value * 2
        return f"{int(converted_value)} pcs"

    else:
        return text

df_combined['Standard_unit'] = df_combined['Standard_unit'].apply(convert_unit)

only_text_units = (
    df_combined['Standard_unit']
    .str.replace(r'[\d\.]+', '', regex=True)
    .str.strip()
)

clean_unit_counts = only_text_units.value_counts(dropna=False)
print(clean_unit_counts)

Standard_unit
kg        3953
ltr       2365
pcs        767
pack        60
mtr         39
bundle       9
page         4
Name: count, dtype: int64


**p. Create & stored data into `Measure_value` and `Measure_unit`:**

In [61]:
df_combined['Measure_value'] = df_combined['Standard_unit'].str.extract(r'(\d*\.?\d+)').astype(float)
df_combined['Measure_unit'] = df_combined['Standard_unit'].str.extract(r'([a-zA-Z]+)')

In [62]:
def categorize_size_specific(row):
    val = row['Measure_value']
    unit = str(row['Measure_unit']).lower().strip()
    
    if unit in ['kg', 'ltr']:
        if val < 1: return 'Small'
        elif 1 <= val <= 2: return 'Medium'
        else: return 'Large'
    
    elif unit in ['pcs', 'pack', 'bundle']:
        if val <= 6: return 'Small'
        elif 6 < val <= 12: return 'Medium'
        else: return 'Large'
        
    elif unit in ['mtr']:
        if val <= 3: return 'Small'
        elif 3 < val <= 10: return 'Medium'
        else: return 'Large'
        
    elif unit in ['page']:
        if val <= 100: return 'Small'
        elif 100 < val <= 500: return 'Medium'
        else: return 'Large'
    return ''

df_combined['Size_Category'] = df_combined.apply(categorize_size_specific, axis=1)


**p. Create `Std_per_unit_price` column:**

In [63]:
df_combined['Std_per_unit_price'] = df_combined['Actual_unit_price'] / df_combined['Measure_value']

q. Create `Size_category` column:

In [64]:
def categorize_size(row):
    val = row['Measure_value']
    unit = row['Measure_unit']
    
    if unit in ['kg', 'ltr']:
        if val <= 1: return 'Small'
        elif val <= 10: return 'Medium'
        else: return 'Large'
        
    elif unit in ['pcs', 'pack', 'bundle']:
        if val <= 5: return 'Small'
        elif val <= 24: return 'Medium'
        else: return 'Large'
        
    elif unit == 'mtr':
        if val <= 2: return 'Small'
        elif val <= 10: return 'Medium'
        else: return 'Large'
    
    elif unit == 'page':
        if val <= 10: return 'Small'   
        elif val <= 100: return 'Medium' 
        else: return 'Large'             
        
    else:
        return 'Small'

df_combined['Size_Category'] = df_combined.apply(categorize_size, axis=1)


**r. In this step, essential products are identified by cleaning the `Product_name` column data and applying category-specific keyword matching to detect relevant items :**


**Step-1:** Cleaning `Product_name` column: Removing Brand name from the text of Product_name column:

In [65]:
def remove_brand(product, brand):
    product = str(product).lower()
    brand_words = str(brand).lower().split()

    for b in brand_words:
        product = re.sub(rf'\b{re.escape(b)}\b', '', product)

    product = re.sub(r'\s+', ' ', product).strip()
    return product

cleaned_product_name = [
    remove_brand(p, b) 
    for p, b in zip(df_combined['Product_name'], df_combined.get('Brand', ''))
]

**Step-2:** Checking frequency of words:

In [66]:
from collections import Counter
from itertools import combinations

subcat_phrase_dict = {}

exclude_categories = ['Fruits', 'Vegetables', 'Meat & Fish']

for subcat, group in df_combined.groupby('Sub-category'):

    if subcat in exclude_categories:
        continue

    phrase_count = Counter()

    for idx in group.index:

        words = cleaned_product_name[idx].split()

        words = list(set([w for w in words if len(w) > 1]))

        for combo in combinations(words, 1):
            phrase = ' '.join(combo)  
            phrase_count[phrase] += 1

    common_phrases = [p for p, c in phrase_count.items() if c >= 5]

    subcat_phrase_dict[subcat] = common_phrases

for k, v in subcat_phrase_dict.items():
    print(f'"{k}": {v},')

"Baby Care": ['milk', 'cereal', 'baby', 'powder', 'cream', 'rice', 'wheat', 'bottle', 'glass', 'feeding', 'feeder', 'smart', 'care', 'nipple', 'silicon', 'shampoo', 'mild', 'gentle', 'oil', 'bath', 'lotion', 'soap', 'olive', 'almond', 'wash', 'parachute', 'face', 'toothpaste', 'orange', 'gel', 'strawberry', 'wipes', 'twinkle', 'savlon', 'jar', 'wet', 'glow', 'milky', 'natural', 'supermom', 'premium', 'brush', 'super', 'diaper', 'pant', 'xl', 'pants', 'xxl', 'nappy', 'happy', 'belt', 'nestle', 'with', 'years)', 'smartcare'],
"Baking & Flour": ['powder', 'soda', 'baking', 'custard', 'orange', 'flour', 'corn', 'food', 'essence', 'colour', 'essential', 'atta', '(dates)', 'khezur', 'premium', 'loose', '(atta)', '(maida)', 'white', '(khejur)', 'dates', 'roasted', 'salt', 'badam', 'light', 'with', 'cashews', 'cake', 'vinegar'],
"Beverages": ['orange', 'powder', 'drink', 'mango', 'instant', 'soft', 'shwapno', 'malt', 'classic', 'speed', 'syrup', 'lemon', 'pineapple', 'seed', 'zero', 'sprite', 

**Step-3:** This step ensures that products in Fruits, Vegetables, and Meat & Fish sub-categories are used as regular products in their original form, while other categories are matched using frequent keywords only :

In [67]:
sub_category_keywords = {
      "Baking & Flour": ['powder', 'flour', 'corn', 'dates', 'khezur', 'badam', 'atta'],
      "Beverages": ['drink', 'mango', 'instant', 'sprite', 'drinks', 'can', 'water', 'coffee', 'fruit', 'apple', 'juice', 'electrolyte', 'drinking', 'tea', 'can'],
      "Breakfast": ['mayonnaise', 'honey', 'spread', 'chocolate', 'oats', 'cereal', 'jelly', 'butter', 'jam'],
      "Canned Food": ['corn', 'olives', 'mushroom'],
      "Chocolates & Icecream": ['chocolate', 'dark', 'milk', 'silk', 'dairy', 'fruit', 'wafer', 'kitkat', 'candy', 'bar', 'ice', 'cream'],
      "Dairy & Eggs": ['powder', 'milk', 'cream', 'ghee', 'dairy', 'cheese', 'uht', 'drink', 'mango', 'chocolate', 'yogurt', 'eggs', 'egg', 'butter'],
      "Frozen Snacks": ['roll', 'spring', 'chicken', 'ball', 'sausage', 'paratha', 'nuggets', 'samosa', 'puri'],
      "Oil": ['mustard', 'sunflower', 'soyabean', 'olive', 'fortified'],
      "Rice": ['rice', 'miniket', 'chinigura', 'basmati'],
      "Salt & Sugar": ['sugar', 'salt'],
      "Sauces & Pickles": ['pickle', 'mango', 'olive', 'mixed', 'tomato', 'sauce', 'ketchup'],
      "Snacks": ['soup', 'chicken', 'noodles', 'masala', 'nuts', 'cracker', 'crackers', 'biscuits', 'chocolate', 'toast', 'biscuit', 'cookies', 'lexus', 'cake', 'sweet', 'chips', 'peanut', 'semai', 'sweets', 'shemai', 'chopstick', 'ramen', 'chanachur', 'pasta', 'macaroni', 'puffed', 'muri', 'badam', 'flattened rice'],
      "Spices & Ingredients": ['turmeric', 'holud', 'pepper', 'coriander', 'dhonia', 'masala', 'cumin', 'chilli', 'morich', 'mix', 'cardamom', 'elachi', 'kismis'],
}

df_combined["Regular_products"] = None

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9 ]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_word_set(text):
    return set(clean_text(text).split())

direct_categories = ["Fruits", "Vegetables", "Meat & Fish"]

for idx, row in df_combined.iterrows():

    sub_cat = row["Sub-category"]

    product_name = row.get("Product_name", "")
    title = row.get("Title", "")

    if sub_cat in direct_categories:

        if pd.notna(product_name) and str(product_name).strip() != "":
            df_combined.at[idx, "Regular_products"] = product_name
        else:
            df_combined.at[idx, "Regular_products"] = title

        continue

    if sub_cat not in sub_category_keywords:
        continue

    keywords = sub_category_keywords[sub_cat]

    product_words = get_word_set(product_name)
    title_words = get_word_set(title)

    found = False

    for kw in keywords:
        kw_set = get_word_set(kw)

        if kw_set.issubset(product_words):
            df_combined.at[idx, "Regular_products"] = product_name
            found = True
            break

    if not found:
        for kw in keywords:
            kw_set = get_word_set(kw)

            if kw_set.issubset(title_words):
                df_combined.at[idx, "Regular_products"] = title
                break

df_combined["Regular_products"] = df_combined["Regular_products"].apply(lambda x: "Yes" if pd.notna(x) else "No")

filled = df_combined["Regular_products"].notna().sum()
blank = df_combined["Regular_products"].isna().sum()

print("🔹 Filled:", filled)
print("🔹 Blank:", blank)

print(df_combined[["Product_name", "Title", "Sub-category", "Regular_products"]].head(20))

🔹 Filled: 7197
🔹 Blank: 0
                            Product_name                                              Title Sub-category Regular_products
0           joldugui anarosh (pineapple)        Joldugui Anarosh (Pineapple) 600gm+ Per Pcs       Fruits              Yes
1                         Malta (Orange)                                     Malta (Orange)       Fruits              Yes
2   Green Apple (Apple Golden Delicious)               Green Apple (Apple Golden Delicious)       Fruits              Yes
3                    dalim (pomegranate)                 Dalim (Pomegranate) 250gm+ Per Pcs       Fruits              Yes
4                    Baby Orange (China)                                Baby Orange (China)       Fruits              Yes
5               Peyara Desi Thai (Guava)                           Peyara Desi Thai (Guava)       Fruits              Yes
6                 Nashpati (Pears White)                             Nashpati (Pears White)       Fruits              Ye

**Step-4 :** Further Checkinng from remaining nulls:

In [68]:
subcat_phrase_dict = {}

exclude_categories = ['Fruits', 'Vegetables', 'Meat & Fish']

filtered_df = df_combined[df_combined["Regular_products"].isna()]

for subcat, group in filtered_df.groupby('Sub-category'):

    if subcat in exclude_categories:
        continue

    phrase_count = Counter()

    for idx in group.index:

        words = cleaned_product_name[idx].split()

        words = list(set([w for w in words if len(w) > 1]))

        for combo in combinations(words, 1):
            phrase = ' '.join(combo)
            phrase_count[phrase] += 1

    common_phrases = [p for p, c in phrase_count.items() if c >= 1]

    subcat_phrase_dict[subcat] = common_phrases

for k, v in subcat_phrase_dict.items():
    print(f'"{k}": {v},')

In [69]:
df_combined[df_combined['Regular_products'].notna()] \
    .groupby('Sub-category')['Regular_products'] \
    .count()

Sub-category
Baby Care                 390
Baking & Flour            190
Beverages                 497
Breakfast                 144
Canned Food                53
Chocolates & Icecream     239
Cleaning-Supplies         454
Dairy & Eggs              259
Frozen Snacks             171
Fruits                     41
Home & Kitchen            203
Lentils & Pulses           57
Meat & Fish               213
Oil                       116
Personal Care            2236
Rice                       85
Salt & Sugar               41
Sauces & Pickles          169
Snacks                    867
Spices & Ingredients      438
Stationeries              188
Vegetables                146
Name: Regular_products, dtype: int64

In [70]:
df_combined[df_combined['Regular_products'].notna()]['Measure_unit'].value_counts()

Measure_unit
kg        3953
ltr       2365
pcs        767
pack        60
mtr         39
bundle       9
page         4
Name: count, dtype: int64

**r. Create Common_brand column :** to identify brands that appear in `both sources` under the same `sub-category` for accurate cross-source comparison.

In [71]:
shwapno = df_combined[df_combined['Source'] == 'Shwapno']
chaldal = df_combined[df_combined['Source'] == 'Chaldal']

common_set = pd.merge(shwapno[['Sub-category', 'Brand']], 
                      chaldal[['Sub-category', 'Brand']], 
                      on=['Sub-category', 'Brand'], 
                      how='inner').drop_duplicates()

common_set['Common_brand'] = 'Yes'

df_combined = pd.merge(df_combined, common_set, on=['Sub-category', 'Brand'], how='left')
df_combined['Common_brand'] = df_combined['Common_brand'].fillna('No')

exclude_brands = ['Non-Brand', 'House Brand']
df_combined.loc[df_combined['Brand'].isin(exclude_brands), 'Common_brand'] = 'No'

print(df_combined)


             Section             Category  Sub-category                                        Title Currency  Previous_price  Sales_price         Unit   Source          Brand  ... Free_product Savings_freeProduct_status Market_price Standard_unit  Measure_value  Measure_unit  Size_Category  Std_per_unit_price Regular_products Common_brand
0         Food Items  Fresh & Perishables        Fruits  Joldugui Anarosh (Pineapple) 600gm+ Per Pcs        ৳            62.0         62.0    Per Piece  Shwapno  Fresh Produce  ...          NaN                                     NaN        0.6 kg            0.6            kg          Small          103.333333              Yes          Yes
1         Food Items  Fresh & Perishables        Fruits                               Malta (Orange)        ৳           299.0        299.0      Per 1kg  Shwapno  Fresh Produce  ...          NaN                                     NaN        1.0 kg            1.0            kg          Small          299.000000     

**s. Create Price_segment Column :**

In [72]:
def price_segment(group):

    q1 = group["Std_per_unit_price"].quantile(0.25)
    q3 = group["Std_per_unit_price"].quantile(0.75)

    def classify(x):
        if x >= q3:
            return "Hight_value"
        elif x <= q1:
            return "Budget_friendly"
        else:
            return "Standard"

    return group["Std_per_unit_price"].apply(classify)

df_combined["Price_Segment"] = df_combined.groupby(
    "Sub-category",
    group_keys=False
).apply(price_segment)

print(df_combined[[
    "Sub-category",
    "Title",
    "Std_per_unit_price",
    "Price_Segment"
]].head(20))

   Sub-category                                              Title  Std_per_unit_price    Price_Segment
0        Fruits        Joldugui Anarosh (Pineapple) 600gm+ Per Pcs          103.333333  Budget_friendly
1        Fruits                                     Malta (Orange)          299.000000         Standard
2        Fruits               Green Apple (Apple Golden Delicious)          395.000000         Standard
3        Fruits                 Dalim (Pomegranate) 250gm+ Per Pcs          615.000000      Hight_value
4        Fruits                                Baby Orange (China)          395.000000         Standard
5        Fruits                           Peyara Desi Thai (Guava)          125.000000         Standard
6        Fruits                             Nashpati (Pears White)          385.000000         Standard
7        Fruits                       Narikel (Coconut) 600gm+ Pcs          253.333333         Standard
8        Fruits  Paka Pepe Deshi Thai (Ripen Papaya) 600gm+ Pcs/

In [74]:
len(df_combined.columns)

29

**t. Save to CSV file for Tableau analysis:** Required columns stored by droping unnecessary columns:

In [80]:
required_columns = [
    'Section', 'Category', 'Sub-category', 'Title',  'Currency', 'Previous_price', 'Market_price', 'Actual_unit_price', 'Sales_price', 'Actual_unit', 'Standard_unit', 'Measure_value', 'Measure_unit', 'Std_per_unit_price', 'Total_savings', 'Free_product', 'Savings_freeProduct_status', 'Price_Segment', 'Size_Category', 'Brand', 'Regular_products', 'Common_brand', 'Source',]

df_final = df_combined[required_columns]

In [81]:
len(df_final.columns)

23

In [82]:
df_final.to_csv('df_final.csv', index='False', encoding='utf-8-sig')

In [83]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 7197 entries, 0 to 7196
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Section                     7197 non-null   str    
 1   Category                    7197 non-null   str    
 2   Sub-category                7197 non-null   str    
 3   Title                       7197 non-null   str    
 4   Currency                    7197 non-null   str    
 5   Previous_price              7197 non-null   float64
 6   Market_price                1212 non-null   float64
 7   Actual_unit_price           7197 non-null   float64
 8   Sales_price                 7197 non-null   float64
 9   Actual_unit                 7197 non-null   str    
 10  Standard_unit               7197 non-null   str    
 11  Measure_value               7197 non-null   float64
 12  Measure_unit                7197 non-null   str    
 13  Std_per_unit_price          7197 non-null   